# 早期强势股 V2：2018—2023 数据质量审计

## tl;dr

- 数据粒度是“周度决策日 × 股票”，155,107 行、307 个决策日、无重复主键。
- V1 数据中 103,492 行没有可执行的 60 日结果；V2 只在其余 51,615 行中构造标签，且经过股票池门禁后实际训练标签为 49,700 行。
- `turnover_20` 在开发期全空，`valuation_percentile` 恒为 0.5；资金因子存在明显覆盖漂移和极端量纲值。
- 发布时间与生效时间审计通过。数据可用于修复后的开发期实验，但不能原样复用 V1 标签和全量特征。

## Context & Methods

### Key Assumptions

- 只读取 V1 已冻结的 2018—2023 年度特征文件；2024、2025 和 2026 不参与本 notebook。
- 决策时点为每个 `asof` 当日 15:00。
- `published_at` 与 `effective_at` 必须不晚于决策时点。
- V2 标签只对 `entry_executable=true` 且 `forward_return_60` 非空的合格股票生成。

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
REPOSITORY_ROOT = Path("D:\\Project\\stock")
SOURCE_PATHS = ['data/research/early_winner_v1/features/ewh_history-75981a3a7bd7495f740b81fa_2018.parquet', 'data/research/early_winner_v1/features/ewh_history-75981a3a7bd7495f740b81fa_2019.parquet', 'data/research/early_winner_v1/features/ewh_history-75981a3a7bd7495f740b81fa_2020.parquet', 'data/research/early_winner_v1/features/ewh_history-75981a3a7bd7495f740b81fa_2021.parquet', 'data/research/early_winner_v1/features/ewh_history-75981a3a7bd7495f740b81fa_2022.parquet', 'data/research/early_winner_v1/features/ewh_history-75981a3a7bd7495f740b81fa_2023.parquet']
frames = [pd.read_parquet(REPOSITORY_ROOT / path) for path in SOURCE_PATHS]
data = pd.concat(frames, ignore_index=True)
data['year'] = pd.to_datetime(data['asof']).dt.year
assert set(data['year']) == set(range(2018, 2024))
assert not data.duplicated(['asof', 'code']).any()
len(data)

155107

## Data

### 1. Confirm grain and yearly volume

In [2]:
year_profile = data.groupby('year').agg(rows=('code','size'), decision_dates=('asof','nunique'), stocks=('code','nunique'), executable_rows=('entry_executable','sum')).reset_index()
display(year_profile)
grain = {'rows': len(data), 'columns': len(data.columns), 'decision_dates': data['asof'].nunique(), 'stocks': data['code'].nunique(), 'duplicate_keys': int(data.duplicated(['asof','code']).sum())}
grain

,year,rows,decision_dates,stocks,executable_rows
0,2018,13187,51,620,3942
1,2019,19048,52,829,6928
2,2020,28885,52,984,9396
3,2021,29845,52,1061,9925
4,2022,32630,50,1225,11019
5,2023,31512,50,1247,10405


{'rows': 155107,
 'columns': 60,
 'decision_dates': 307,
 'stocks': 1441,
 'duplicate_keys': 0}

## Results

### 2. Label scope is the highest-impact V1 issue

In [3]:
outcome = pd.to_numeric(data['forward_return_60'], errors='coerce')
label_scope = pd.crosstab(data['entry_executable'], outcome.notna(), margins=True)
display(label_scope)
assert ((~data['entry_executable']) == outcome.isna()).all()
{'rows_without_outcome': int(outcome.isna().sum()), 'rows_with_outcome': int(outcome.notna().sum())}

forward_return_60,False,True,All
entry_executable,,,
False,103492,0,103492
True,0,51615,51615
All,103492,51615,155107


{'rows_without_outcome': 103492, 'rows_with_outcome': 51615}

### 3. Missingness and distribution drift remove several features from V2

In [4]:
feature_columns = ['northbound_change_ratio','institution_holding_change_ratio','turnover_20','valuation_percentile']
missing = data.groupby('year')[feature_columns].agg(lambda s: pd.to_numeric(s, errors='coerce').isna().mean()).round(4)
display(missing)
distribution = pd.DataFrame({column: {'nonmissing_rate': pd.to_numeric(data[column], errors='coerce').notna().mean(), 'unique': pd.to_numeric(data[column], errors='coerce').nunique(dropna=True), 'p01': pd.to_numeric(data[column], errors='coerce').quantile(.01), 'p99': pd.to_numeric(data[column], errors='coerce').quantile(.99), 'maximum': pd.to_numeric(data[column], errors='coerce').max()} for column in feature_columns}).T
display(distribution)

,northbound_change_ratio,institution_holding_change_ratio,turnover_20,valuation_percentile
year,,,,
2018,0.1903,0.5248,1.0,0.0
2019,0.2041,0.0068,1.0,0.0
2020,0.2759,0.0115,1.0,0.0
2021,0.2420,0.0081,1.0,0.0
2022,0.2679,0.0046,1.0,0.0
2023,0.1784,0.0053,1.0,0.0


,nonmissing_rate,unique,p01,p99,maximum
northbound_change_ratio,0.768212,105210.0,-0.785004,8.703687,2206774.00
institution_holding_change_ratio,0.948822,15051.0,-0.977498,32.843125,210981.24
turnover_20,0.000000,0.0,NaN,NaN,NaN
valuation_percentile,1.000000,1.0,0.500000,0.500000,0.50


### 4. Point-in-time timestamps pass their hard audit

In [5]:
decision = pd.to_datetime(data['asof']) + pd.Timedelta(hours=15)
published = pd.to_datetime(data['published_at'], errors='coerce')
effective = pd.to_datetime(data['effective_at'], errors='coerce')
time_audit = {'published_missing': int(published.isna().sum()), 'published_after_decision': int((published > decision).sum()), 'effective_missing': int(effective.isna().sum()), 'effective_after_decision': int((effective > decision).sum())}
assert all(value == 0 for value in time_audit.values())
time_audit

{'published_missing': 0,
 'published_after_decision': 0,
 'effective_missing': 0,
 'effective_after_decision': 0}

## Takeaways

1. **High severity / high confidence:** V2 必须排除无执行结果的行，不能把它们默认标为 0。
2. **High severity / high confidence:** 每个训练折都应删除全空、近空和常量特征；截尾与中位数只能由训练折计算。
3. **High severity / medium confidence:** 资金因子需要独立核对量纲和历史覆盖；在完成前，技术/行业核心方案不依赖它们。
4. **通过项:** 点时连接与主键粒度可以作为 V2 开发期输入。
5. **边界:** 本审计不证明策略有效，也没有打开 2024/2025 或 2026。